# Importing Libraries

In [ ]:
# Import polars for analytics
import polars as pl

# Path for data folder
from pathlib import Path

# Define Data Path

In [ ]:
# Define data folder
data_folder = Path("../data")

# Loading the Dataset

In [ ]:
# Load cleaned dataset
sales_df = pl.read_parquet(data_folder /"sales_cleaned.parquet")

# Preview dataset
sales_df.head()

# Product Performance

In [ ]:
# Calculate product performance
product_performance = (
    sales_df.group_by(["product_id","product_name","category"])
    .agg(
        # number of transactions
        pl.len().alias("transactions"),

        # total revenue
        pl.col("revenue").sum().alias("total_revenue"),

        # average product price
        pl.col("revenue").mean().alias("avg_price")

    )
)

product_performance.head()

# Sort Top and Bottom Products

In [ ]:
# Sort top products by revenue
top_products = (product_performance.sort("total_revenue",descending=True))

top_products.head()

In [ ]:
# Sort Bottom products by revenue
bottom_products = (product_performance.sort("total_revenue",descending=False))

bottom_products.head()

# Category Performance

In [ ]:
# Revenue by category
category_performance = (
    sales_df.group_by(["category"])
    .agg(
        pl.len().alias("transactions"),
        
        pl.col("revenue").sum().alias("total_revenue"),

        pl.col("revenue").mean().alias("avg_order_value")
    )
    .sort("total_revenue", descending=True)
)

category_performance.sort("total_revenue",descending = True).head()

# City Performance

In [ ]:
# Revenue by city
city_performance = (
    sales_df.group_by(["city"])
    .agg(
        pl.len().alias("transactions"),
        
        pl.col("revenue").sum().alias("total_revenue"),

        pl.col("revenue").mean().alias("avg_order_value")
    )
    .sort("total_revenue", descending=True)
)

city_performance.sort("avg_order_value",descending = True).head()

In [ ]:
city_performance.sort("total_revenue",descending = True).head()

In [ ]:
city_performance.sort("transactions",descending = True).head()

## Pareto Analysis (80/20 Rule)
Checking wether few products drive most revenue

In [ ]:
# Add revenue share 
products_pareto = (
    top_products.with_columns(
        (pl.col("total_revenue") / 
        pl.col("total_revenue").sum()
        ).alias("revenue_share")
    )
)

products_pareto.sort("revenue_share",descending = True)

## Revenue Share

In [ ]:
product_performance = product_performance.with_columns(
    (pl.col("total_revenue") / pl.col("total_revenue").sum()*100).round(2).alias("revenue_share_pct" )
)

product_performance.sort("revenue_share_pct",descending = True)

# Adding cumulative column to the product table

In [ ]:
# Sort products by revenue
product_performance = product_performance.sort(
    "total_revenue",descending=True    
)

# Add cumulative revenue column
product_performance = product_performance.with_columns(

    # cumulative revenue
    pl.col("total_revenue").cum_sum().alias("cumulative_revenue"),

    # cumulative revenue percentage
    (
        pl.col("total_revenue").cum_sum()
        /
        pl.col("total_revenue").sum()
        * 100
    ).round(2).alias("cumulative_revenue_pct")
)

# view table
product_performance


## Save the Datasets

In [ ]:
# Save product dataset
product_performance.write_parquet(data_folder/"product_performance.parquet")

# Save category dataset
category_performance.write_parquet(data_folder/"category_performance.parquet")

# Save city dataset
city_performance.write_parquet(data_folder/"city_performance.parquet")

# Save top products dataset
top_products.write_parquet(data_folder/"top_products.parquet")

In [ ]:
top_products.head(10)

In [ ]:
category_performance

In [ ]:
product_performance.select(pl.col("total_revenue").sum())